# R8 v12 | Demo on two real PDFs

Private Kaggle notebook. Enable GPU T4 x2 and Internet. Attach the private dataset containing research-rag-kaggle-v2.zip. Run Cells 1 to 4. This uses the locked R8 v12 verifier and audited table coordinates for the two input PDFs. Single-cell claims have a coordinate-based evidence check inside the R8 adapter. Gold is kept outside this notebook.


## 1. Khởi tạo workspace và kiểm checkpoint

In [ ]:
RUNTIME_SHA256 = 'f22413f085e4cf4da0c93ba13fc544e8c65d3d4bbed1d1c6d1c13775eaaba43a'
PAYLOAD_SHA256 = 'c2f6decea7021c83a99a7cb4e4985c4d511723fdebb4c6066a0d97e4523fbaa5'
ASSETS = {'05_writer/checkpoints/writer_qlora/adapter_config.json': {'sha256': 'ca0c0487fcd867c57881b4a820a1fa909964a7799221b242475e05103ca7090d', 'size': 838}, '05_writer/checkpoints/writer_qlora/adapter_model.safetensors': {'sha256': '2c233996b954baa8cb9b19d55e586e3ab59db9beeb7e80325f5d82f69d8d2a27', 'size': 73911112}, '05_writer/checkpoints/writer_qlora/added_tokens.json': {'sha256': '58b54bbe36fc752f79a24a271ef66a0a0830054b4dfad94bde757d851968060b', 'size': 605}, '05_writer/checkpoints/writer_qlora/merges.txt': {'sha256': '8831e4f1a044471340f7c0a83d7bd71306a5b867e95fd870f74d0c5308a904d5', 'size': 1671853}, '05_writer/checkpoints/writer_qlora/special_tokens_map.json': {'sha256': '76862e765266b85aa9459767e33cbaf13970f327a0e88d1c65846c2ddd3a1ecd', 'size': 613}, '05_writer/checkpoints/writer_qlora/tokenizer.json': {'sha256': '9c5ae00e602b8860cbd784ba82a8aa14e8feecec692e7076590d014d7b7fdafa', 'size': 11421896}, '05_writer/checkpoints/writer_qlora/tokenizer_config.json': {'sha256': '57fea41fa99fce34f91450643e48b3bfbb078c32c6831a3c06834f122c200510', 'size': 7336}, '05_writer/checkpoints/writer_qlora/vocab.json': {'sha256': 'ca10d7e9fb3ed18575dd1e277a2579c16d108e32f27439684afa0e10b1440910', 'size': 2776833}, 'train_agent3_summarizer/checkpoints/critic_qlora/final/adapter_config.json': {'sha256': '449eae6ce9dc58c5c783f99a89242193f18c06b405ab9a86b9a44b3343b6ddc7', 'size': 1054}, 'train_agent3_summarizer/checkpoints/critic_qlora/final/adapter_model.safetensors': {'sha256': 'd3123dcea7298404133b0db359f533ceb2f4334a52ed93dcb11331e5de4d0c06', 'size': 40400200}, 'train_agent3_summarizer/checkpoints/critic_qlora/final/chat_template.jinja': {'sha256': 'cd8e9439f0570856fd70470bf8889ebd8b5d1107207f67a5efb46e342330527f', 'size': 2507}, 'train_agent3_summarizer/checkpoints/critic_qlora/final/tokenizer.json': {'sha256': '3fd169731d2cbde95e10bf356d66d5997fd885dd8dbb6fb4684da3f23b2585d8', 'size': 11421892}, 'train_agent3_summarizer/checkpoints/critic_qlora/final/tokenizer_config.json': {'sha256': '47bfa3e7727312946b29ac10d6dd0672d63cf7815b2a160b9523872040d2e536', 'size': 665}, 'train_agent3_summarizer/checkpoints/generator_qlora/final/adapter_config.json': {'sha256': 'b3f8aaf9cbfba12f590fbe96f22b3acb2c3a647c866ed0f5aef7edcdb15285d7', 'size': 1103}, 'train_agent3_summarizer/checkpoints/generator_qlora/final/adapter_model.safetensors': {'sha256': 'bb844bd730bfb1aa49afaedd6c6ee4c66b5553966eec7c3ea443d32c40352b7c', 'size': 161533192}, 'train_agent3_summarizer/checkpoints/generator_qlora/final/chat_template.jinja': {'sha256': 'cd8e9439f0570856fd70470bf8889ebd8b5d1107207f67a5efb46e342330527f', 'size': 2507}, 'train_agent3_summarizer/checkpoints/generator_qlora/final/tokenizer.json': {'sha256': '3fd169731d2cbde95e10bf356d66d5997fd885dd8dbb6fb4684da3f23b2585d8', 'size': 11421892}, 'train_agent3_summarizer/checkpoints/generator_qlora/final/tokenizer_config.json': {'sha256': '47bfa3e7727312946b29ac10d6dd0672d63cf7815b2a160b9523872040d2e536', 'size': 665}}
RUNTIME_B64 = '...'

from pathlib import Path
from io import BytesIO
from zipfile import ZipFile
import base64, hashlib, os, shutil, subprocess, sys, tempfile

subprocess.run(["nvidia-smi", "-L"], check=True)
input_dir = Path("/kaggle/input")
session = Path(tempfile.mkdtemp(prefix="r8-v12-e2e-", dir="/tmp"))
root = session / "workspace"
root.mkdir()
runtime = base64.b64decode(RUNTIME_B64)
assert hashlib.sha256(runtime).hexdigest() == RUNTIME_SHA256, "Gói runtime sai hash"
with ZipFile(BytesIO(runtime)) as bundle:
    bundle.extractall(root)
payload = root / "artifacts/atomic-cellfix29-e2e-quantity-candidate.zip"
assert hashlib.sha256(payload.read_bytes()).hexdigest() == PAYLOAD_SHA256
candidate = root / "research-runs/atomic-cellfix29-candidate/workspace"
with ZipFile(payload) as bundle:
    bundle.extractall(candidate)

anchor = "train_agent3_summarizer/checkpoints/generator_qlora/final/adapter_config.json"
found = []
for path in input_dir.rglob("adapter_config.json"):
    if path.as_posix().endswith("/" + anchor):
        top = path.parents[len(Path(anchor).parts) - 1]
        if all((top / rel).is_file() for rel in ASSETS):
            found.append(top)
assert len(found) <= 1, "Có nhiều workspace checkpoint. Chỉ giữ một input phù hợp."
def verify(path, expected):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    assert path.stat().st_size == expected["size"] and digest.hexdigest() == expected["sha256"], path

if found:
    print("Checkpoint input", found[0])
    for rel, expected in ASSETS.items():
        source = found[0] / rel
        verify(source, expected)
        target = root / rel
        target.parent.mkdir(parents=True, exist_ok=True)
        target.symlink_to(source)
else:
    matches = list(input_dir.rglob("research-rag-kaggle-v2.zip"))
    assert len(matches) == 1, "Thêm dataset Private research-rag-five-agents-workspace hoặc gói v2 đúng phiên bản."
    print("Checkpoint archive", matches[0])
    with ZipFile(matches[0]) as checkpoint_zip:
        for rel, expected in ASSETS.items():
            target = root / rel
            target.parent.mkdir(parents=True, exist_ok=True)
            with checkpoint_zip.open(rel) as source, target.open("wb") as out:
                shutil.copyfileobj(source, out)
            verify(target, expected)

print("Workspace", root)
print("Checkpoint và runtime đã khớp hash")
print("Disk /tmp còn trống (GiB)", round(shutil.disk_usage("/tmp").free / 2**30, 1))


## 2. Dùng lại CUDA/PyTorch của Kaggle và chạy preflight

Chờ **CELL 2 PASS**. Nếu lỗi, tải thư mục `cellfix29-cell2-*` trong Output để kiểm log.

Cell 2 checks HTTPS access to PyPI for dependencies and Hugging Face for model downloads. If Kaggle starts a new session after you enable Internet, start again at Cell 1.


In [ ]:
# Paste into Cell 2 after running Cell 1 of the R8 v12 E2E notebook.
import json
import os
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

setup_dir = Path(tempfile.mkdtemp(prefix="r8-v12-cell2-", dir="/kaggle/working"))
log_path = setup_dir / "cell2-setup.log"
status_path = setup_dir / "cell2-status.json"
venv = Path(tempfile.mkdtemp(prefix="venv-kaggle-", dir=session))
python = str(venv / "bin/python")
env = {**os.environ, "PYTHONPATH": str(root), "HF_HOME": "/tmp/r8-v12-e2e-hf"}


def setup_status(stage, state, **details):
    status_path.write_text(json.dumps({
        "stage": stage, "state": state,
        "tmp_free_gib": round(shutil.disk_usage(session).free / 2**30, 2),
        **details,
    }, indent=2), encoding="utf-8")


def run_setup(stage, command):
    setup_status(stage, "running")
    print(f"Cell 2 · {stage}", flush=True)
    with log_path.open("a", encoding="utf-8") as log:
        print(f"\n=== {stage} ===", file=log, flush=True)
        completed = subprocess.run(command, cwd=root, env=env, stdout=log, stderr=subprocess.STDOUT)
    if completed.returncode:
        setup_status(stage, "failed", exit_code=completed.returncode)
        print("\n".join(log_path.read_text(encoding="utf-8", errors="replace").splitlines()[-18:]))
        raise RuntimeError(f"{stage} exited {completed.returncode}. See {log_path}")
    setup_status(stage, "completed")


print("Cell 2 setup files", setup_dir, flush=True)
run_setup("gpu", ["nvidia-smi"])
run_setup("reuse-kaggle-python", [
    sys.executable, "-m", "venv", "--without-pip", "--system-site-packages", str(venv),
])

base_info = setup_dir / "cuda-base.json"
probe = """
from importlib import metadata
import json, re, sys
from pathlib import Path
import torch

assert torch.version.cuda and torch.cuda.is_available(), "Kaggle PyTorch cannot access CUDA"
assert torch.ones(1, device="cuda:0").item() == 1
names = {re.sub(r"[-_.]+", "-", d.metadata["Name"]).lower()
         for d in metadata.distributions() if d.metadata.get("Name")}
protected = {n: metadata.version(n) for n in sorted(names)
             if n in {"torch", "torchvision", "torchaudio", "triton", "pytorch-triton"}
             or n.startswith(("nvidia-", "cuda-"))}
info = dict(python=sys.version, torch=torch.__version__, cuda=torch.version.cuda,
            torch_path=str(Path(torch.__file__).resolve()), packages=protected,
            gpus=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
Path(sys.argv[1]).write_text(json.dumps(info, indent=2))
print(json.dumps(info, indent=2))
"""
run_setup("base-cuda-check", [python, "-u", "-c", probe, str(base_info)])
base = json.loads(base_info.read_text())
print(f"Reusing Kaggle torch {base['torch']} · CUDA {base['cuda']}", flush=True)

network_probe = """
import socket, ssl, sys
from urllib.error import URLError
from urllib.request import urlopen

checks = {
    "PyPI": "https://pypi.org/simple/langgraph/",
    "Hugging Face": "https://huggingface.co/api/models/Qwen/Qwen2.5-7B-Instruct",
}
failed = False
for service, url in checks.items():
    try:
        with urlopen(url, timeout=10) as response:
            print(f"{service}: HTTP {response.status}")
    except (URLError, OSError) as exc:
        reason = exc.reason if isinstance(exc, URLError) else exc
        if isinstance(reason, socket.gaierror):
            kind = "DNS resolution failed"
        elif isinstance(reason, ssl.SSLError):
            kind = "HTTPS certificate or TLS error"
        else:
            kind = "HTTPS request failed"
        print(f"{service}: {kind} ({exc})", file=sys.stderr)
        failed = True
if failed:
    sys.exit("Enable Internet in Kaggle Notebook Settings. If already enabled, restart "
             "the session and check Internet again. Run Cell 1 in any new session, then Cell 2.")
"""
run_setup("check-network", [python, "-u", "-c", network_probe])

constraints = setup_dir / "cuda-constraints.txt"
constraints.write_text("".join(f"{name}=={version}\n" for name, version in base["packages"].items()))
report = setup_dir / "pip-plan.json"
pip = [python, "-m", "pip", "--isolated", "install", "--disable-pip-version-check",
       "--progress-bar", "off", "--only-binary=:all:"]
run_setup("resolve-project-dependencies", [
    *pip, "--dry-run", "--report", str(report), "-c", str(constraints),
    "-r", "requirements-orchestrator.txt", "-r", "requirements-atomic-verifier.txt",
])
planned = json.loads(report.read_text())["install"]
import re

names = {re.sub(r"[-_.]+", "-", item["metadata"]["name"]).lower() for item in planned}
blocked = {n for n in names if n in base["packages"]
           or n in {"torch", "torchvision", "torchaudio", "triton", "pytorch-triton"}
           or n.startswith(("nvidia-", "cuda-"))}
if blocked:
    setup_status("resolve-project-dependencies", "blocked", protected_packages=sorted(blocked))
    raise RuntimeError(f"Install plan touches the Kaggle CUDA stack: {sorted(blocked)}. See {report}")
if planned:
    pins = [f"{item['metadata']['name']}=={item['metadata']['version']}" for item in planned]
    # The dry run resolves transitive dependencies. Install only that approved package list.
    run_setup("install-approved-packages", [*pip, "--no-deps", *pins])

preflight = """
from importlib import metadata
import json, sys
from pathlib import Path
from packaging.requirements import Requirement
import torch
from bitsandbytes.nn import Linear4bit
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from sentence_transformers import SentenceTransformer
import faiss
from research_rag.r8_adapter import R8_POLICY_ID, load_r8_runtime
from research_rag.adapters import PdfReader

expected = json.loads(Path(sys.argv[1]).read_text())
assert str(Path(torch.__file__).resolve()) == expected["torch_path"]
assert torch.__version__ == expected["torch"] and torch.version.cuda == expected["cuda"]
assert all(metadata.version(name) == version for name, version in expected["packages"].items())
for file in ("requirements-orchestrator.txt", "requirements-atomic-verifier.txt"):
    for line in Path(file).read_text().splitlines():
        spec = line.split("#", 1)[0].strip()
        if spec:
            req = Requirement(spec)
            version = metadata.version(req.name)
            assert req.specifier.contains(version), f"{req.name}={version} differs from {spec}"

with torch.inference_mode():
    layer = Linear4bit(64, 64, bias=False, compute_dtype=torch.float16, quant_type="nf4").to("cuda:0")
    x = torch.ones((1, 64), device="cuda:0", dtype=torch.float16)
    assert torch.isfinite(layer(x)).all().item(), "NF4 forward failed"
runtime = load_r8_runtime()
assert R8_POLICY_ID == "r8-v12:9adedbb5eb5269a41afb4a0f3a66275e079f49a0772090138c20124858279edd"
chunks = PdfReader().read("research_rag/evaluation/pdfs/simple_text.pdf")
assert any("The Aster trial enrolled 48 participants." in " ".join(c["text"].split())
           and c["metadata"]["page_no"] == 0 for c in chunks)
print("CUDA and NF4 passed")
print("Policy", R8_POLICY_ID)
print("PDF evidence is present on the first page")
"""
run_setup("cuda-nf4-r8-v12-pdf-preflight", [python, "-u", "-c", preflight, str(base_info)])
setup_status("cell2", "passed", torch=base["torch"], cuda=base["cuda"])
print("CELL 2 PASS · CUDA/NF4, R8 v12 và PDF. Có thể chạy Cell 3.")


## 3. Run three PDF questions


In [ ]:
LOCK_JSON = '{\n  "role": "r8_v12_real_pdf_demo_table_proof",\n  "inputs_sha256": "7ce8aad17ff703405b64ccd5a704d0139e1d71d928a54e017532fea8995cc297",\n  "gold_sha256": "e7b25b24ec60118d11f2bdf54791e8618b7e528d7be62a04d729d6e7cf23fda5",\n  "notebook_template": "scripts/kaggle_r8_v12_e2e_smoke.ipynb",\n  "notebook_template_sha256": "ee10bce2d4bd6c2a741677e8f5cf00da08d370758923942034e74ab54b2f73e7",\n  "runtime_sha256": "f22413f085e4cf4da0c93ba13fc544e8c65d3d4bbed1d1c6d1c13775eaaba43a",\n  "baseline_lock_sha256": "9adedbb5eb5269a41afb4a0f3a66275e079f49a0772090138c20124858279edd",\n  "smoke_zip_sha256": "486f8c877eab25afdbb5fddf4c6674829fd3d547438f77d787a3024f0ad75042",\n  "retrieval": "dense",\n  "initial_top_k": 5,\n  "checkpoint_archive": "research-rag-kaggle-v2.zip",\n  "pdfs": {\n    "kaggledbqa": {\n      "path": "research-runs/r8-v12-support-guard/table-coordinate-cross-paper-v1/sources/kaggledbqa.pdf",\n      "sha256": "c0fc149226f39717947f8e965fc67c974158c1d8943aac00ae66b1fdc558e753",\n      "url": "https://aclanthology.org/2021.acl-long.176.pdf#page=7"\n    },\n    "performance-critical-tokens": {\n      "path": "research-runs/r8-v12-support-guard/table-coordinate-cross-paper-v1/sources/performance-critical-tokens.pdf",\n      "sha256": "f25125f6107e47cdca916cd02dfc633f020b1d73ddd91ade8169185ec42bfcbe",\n      "url": "https://arxiv.org/pdf/2401.11323v4#page=5"\n    }\n  },\n  "base_runtime_sha256": "047052066632248f84ec88f368b33fee9ceb22705bcc95a6150c5a73a9ecc79e",\n  "reader_max_pages": 26,\n  "reader_runtime_members": {\n    "research_rag/adapters.py": "ac24d6b312efaa71591b2b2962934ad644bf12c065d1063bd9bf909979a46dc0",\n    "research_rag/graph.py": "c67d7a2dafe16db1836f3da6404aebb544c84d256c42da6694e046a1dd906765",\n    "research_rag/local.py": "6c62352449d3ad989d8bc21859171fc3f3d466af21a99aa0883da5d3cf77c473",\n    "research_rag/pdf_cell_context.py": "ad3d6ba985f4224c8fbfedd30f60d51b4d986a619f72c746e54e8cf7d98df1af",\n    "artifacts/r8-v12-e2e-real-pdf/table-coordinates.json": "320f8cca39bfbe6c04370c803036518be7a034fbc4b48addf038bc6c7ff4ff9c",\n    "research_rag/r8_adapter.py": "1bad2641347f4e95b17d5f2c22107964c09c6695da1b160de2fd503cf12a2ab4"\n  }\n}\n'
LOCK_SHA256 = 'd27f5dda39f3b616aa8fda405ce2980fc2cb0e2aa9790517a8efc88124a24591'
INPUTS_JSON = '[\n  {\n    "id": "fires-exact-match",\n    "paper": "kaggledbqa",\n    "question": "Without fine-tuning, what exact-match accuracy did RAT-SQL achieve on the Fires database in Table 4?"\n  },\n  {\n    "id": "temp-sst2-accuracy",\n    "paper": "performance-critical-tokens",\n    "question": "In the representation-level +TEMP ablation in Table 2, what SST2 accuracy is reported for Llama 2 7B?"\n  },\n  {\n    "id": "missing-seeds",\n    "paper": "kaggledbqa",\n    "question": "What were the three exact random seed integers used for the KaggleDBQA RAT-SQL experimental runs summarized in Table 4?"\n  }\n]\n'
PDFS = {'kaggledbqa': '...', 'performance-critical-tokens': '...'}

import json

assert json.loads(status_path.read_text())["state"] == "passed", "Run Cell 2 first."

def check_bytes(raw, expected, what):
    if hashlib.sha256(raw).hexdigest() != expected:
        raise ValueError(f"{what}: SHA-256 mismatch")

lock_bytes = LOCK_JSON.encode("utf-8")
input_bytes = INPUTS_JSON.encode("utf-8")
check_bytes(lock_bytes, LOCK_SHA256, "Demo manifest")
lock = json.loads(lock_bytes)
check_bytes(input_bytes, lock["inputs_sha256"], "Questions")
assert lock["runtime_sha256"] == RUNTIME_SHA256
check_bytes((root / "artifacts/factcheck-verdict-r8-v12-holdout/candidate-lock.json").read_bytes(),
            lock["baseline_lock_sha256"], "R8 verifier lock")
for name, expected in lock["reader_runtime_members"].items():
    check_bytes((root / name).read_bytes(), expected, name)
cases = json.loads(input_bytes)
assert len(cases) == len({case["id"] for case in cases}) == 3
assert set(PDFS) == set(lock["pdfs"]) == {case["paper"] for case in cases}
input_dir = root / "real_pdf_inputs"
input_dir.mkdir(exist_ok=True)
pdf_paths = {}
for paper, encoded in PDFS.items():
    raw = base64.b64decode(encoded, validate=True)
    check_bytes(raw, lock["pdfs"][paper]["sha256"], paper)
    pdf_paths[paper] = input_dir / (paper + ".pdf")
    pdf_paths[paper].write_bytes(raw)

out = Path(tempfile.mkdtemp(prefix="r8-v12-e2e-real-pdf-table-proof-", dir="/kaggle/working"))
(out / "manifest.json").write_bytes(lock_bytes)
(out / "inputs.json").write_bytes(input_bytes)
(out / "provenance.json").write_text(json.dumps({
    "manifest_sha256": LOCK_SHA256,
    "runtime_sha256": RUNTIME_SHA256,
    "verifier_lock_sha256": lock["baseline_lock_sha256"],
    "checkpoint_members": ASSETS,
    "environment": json.loads(base_info.read_text()),
}, indent=2), encoding="utf-8")

for case in cases:
    case_dir = out / case["id"]
    case_dir.mkdir()
    command = [
        python, "-u", "-m", "research_rag.local",
        "--verifier", "r8-v12", "--question", case["question"], "--pdf", str(pdf_paths[case["paper"]]),
        "--pdf-max-pages", str(lock["reader_max_pages"]),
        "--pdf-table-coordinates", str(root / 'artifacts/r8-v12-e2e-real-pdf/table-coordinates.json'),
        "--retrieval-policy", lock["retrieval"], "--initial-top-k", str(lock["initial_top_k"]),
        "--quantize-4bit", "--online-model-downloads", "--profile-resources",
        "--process-timeout", "3600", "--metrics-output", str(case_dir / "metrics.json"),
        "--events-output", str(case_dir / "events.jsonl"), "--trace-output", str(case_dir / "trace.private.jsonl"),
    ]
    result = {"id": case["id"], "exit_code": None, "error_type": None}
    try:
        with (case_dir / "stdout.log").open("w", encoding="utf-8") as stdout,              (case_dir / "stderr.log").open("w", encoding="utf-8") as stderr:
            proc = subprocess.run(command, cwd=root, env=env, stdout=stdout, stderr=stderr, timeout=3720)
        result["exit_code"] = proc.returncode
    except (OSError, subprocess.TimeoutExpired) as error:
        result["error_type"] = type(error).__name__
    pending = case_dir / "process.tmp"
    pending.write_text(json.dumps(result, indent=2), encoding="utf-8")
    pending.replace(case_dir / "process.json")
    print(case["id"], result["exit_code"], result["error_type"], flush=True)


## 4. Package the three cases for review


In [ ]:

def read_json(path, *, lines=False):
    if not path.is_file():
        return [] if lines else {}
    raw = path.read_text(encoding="utf-8")
    return [json.loads(line) for line in raw.splitlines() if line.strip()] if lines else json.loads(raw)

reports = []
for case in cases:
    case_dir = out / case["id"]
    error = None
    try:
        process = read_json(case_dir / "process.json")
        trace = read_json(case_dir / "trace.private.jsonl", lines=True)
    except (OSError, ValueError) as exc:
        process, trace, error = {}, [], type(exc).__name__
    state = next((e["state"] for e in reversed(trace) if e.get("event") == "graph_result"), {})
    report = state.get("final_report")
    claims = state.get("verified_claims", [])
    chunks = {item["source_id"]: item for item in state.get("pdf_chunks", [])}
    cited = []
    for claim in claims:
        source = chunks.get(claim["source_id"], {})
        p0 = source.get("metadata", {}).get("page_no")
        cited.append({**claim, "pdf_page": p0 + 1 if isinstance(p0, int) else None,
                      "pdf_source": source.get("text"),
                      "pdf_source_metadata": source.get("metadata")})
    status = "returned" if process.get("exit_code") == 0 and report else "failed_or_missing"
    reports.append({"id": case["id"], "paper": case["paper"], "question": case["question"], "run_status": status,
                    "process": process, "graph_status": state.get("status"), "errors": state.get("errors", []),
                    "export_error": error, "report": report, "citations": cited,
                    "r8_reviews": [e for e in trace if e.get("event") == "r8_verifier_result"]})

complete = len(reports) == 3 and all(
    row["process"].get("id") == row["id"] and
    (isinstance(row["process"].get("exit_code"), int) or row["process"].get("error_type"))
    for row in reports
)
summary = {"case_count": len(reports), "complete_roster": complete,
           "returned_reports": sum(r["run_status"] == "returned" for r in reports),
           "quality_review": "pending", "cases": reports}
(out / "review.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
archive = out.with_name(out.name + (".zip" if complete else "-incomplete.zip"))
temporary_archive = archive.with_suffix(".zip.tmp")
with ZipFile(temporary_archive, "w", compression=8) as bundle:
    for path in sorted(out.rglob("*")):
        if path.is_file():
            bundle.write(path, path.relative_to(out).as_posix())
temporary_archive.replace(archive)
print("ZIP to send:", archive)
print("Returned reports:", summary["returned_reports"], "/ 3")
